# Feature Track 4d: RAG in an Agent

Notebook 4c ran the retrieval tool loop manually. Here the same retrieval tool is plugged into a `ToolAgent`, which handles the loop automatically. The result is a clean, reusable RAG agent that retrieves only when needed and maintains conversation history across turns.

---

## Setup

**Prerequisites:** `conversational_toolkit` and `backend` installed in editable mode. Set `OPENAI_API_KEY`.

In [ ]:
from conversational_toolkit.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from conversational_toolkit.llms.base import LLMMessage, Roles, MessageContent
from conversational_toolkit.llms.openai import OpenAILLM
from conversational_toolkit.vectorstores.chromadb import ChromaDBVectorStore


from conversational_toolkit.retriever.vectorstore_retriever import VectorStoreRetriever
from conversational_toolkit.agents.tool_agent import ToolAgent, QueryWithContext

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    build_llm,
    build_vector_store,
    VS_PATH,
    EMBEDDING_MODEL,
)

from sme_kt_zh_collaboration_rag.feature4_tool_agents import RetrieveRelevantChunks


BACKEND = "openai"  # "ollama" or "openai"
FORCE_REBUILD = False  # set True to re-embed from scratch

if "sentence-transformers" in EMBEDDING_MODEL:
    embedding_model = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL)
else:
    embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)

if FORCE_REBUILD or not VS_PATH.exists():
    chunks = load_chunks(max_files=5)
    chunks = [c for c in chunks if c.mime_type.startswith("text")]
    db_chroma = await build_vector_store(
        chunks, embedding_model, db_path=VS_PATH, reset=FORCE_REBUILD
    )
else:
    # Vector store already exists -> open it without re-embedding
    db_chroma = ChromaDBVectorStore(db_path=str(VS_PATH))
    print(
        f"Reusing existing vector store at {VS_PATH} ({db_chroma.collection.count()} chunks)"
    )

vector_store = VectorStoreRetriever(embedding_model, db_chroma, top_k=5)

if not BACKEND:
    raise ValueError('Set BACKEND to "ollama" or "openai" before running.')

llm = build_llm(backend=BACKEND)

In [ ]:
retriever_tool = RetrieveRelevantChunks(
    name="retrieve_relevant_chunks",
    description="Retrieves the most relevant chunks based on a query.",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The query to retrieve relevant chunks for.",
            },
        },
        "required": ["query"],
        "additionalProperties": False,
    },
    retriever=vector_store,
)

---

## Define the Agent

The retrieval tool is passed to the LLM, which is then wrapped in a `ToolAgent`. The system prompt tells the agent to trust and cite tool results when it uses them.

In [ ]:
llm = OpenAILLM(tool_choice="auto", tools=[retriever_tool])

# Define the prompt
prompt = "You are a helpful assistant, answer shortly. Use the tools only when they are relevant, but if you do so, trust the results from the tools and use them in your answer, cite them precisely if you use them."
prompt_as_message = LLMMessage(
    content=[MessageContent(type="text", text=prompt)], role=Roles.SYSTEM
)

rag_agent = ToolAgent(system_prompt=prompt, llm=llm, max_steps=5)

## Test the Agent

Two queries run sequentially and are added to `conversation`. The first (general knowledge) should not trigger the retrieval tool; the second (domain-specific) should.

The agent handles the full tool loop internally. The returned `answer` is always the final LLM response, after any tool calls have already been executed and fed back. Intermediate tool calls are not visible in the output. You can only see them in the `DEBUG` logs.


### First Simple Question

In [ ]:
conversation: list[LLMMessage] = [prompt_as_message]

query_simple = "What is a Einstein theory of relativity in the context of physics?"
query_simple_as_message = LLMMessage(
    content=[MessageContent(type="text", text=query_simple)], role=Roles.USER
)
query_simple_with_context = QueryWithContext(query=query_simple, history=[])

answer_simple = await rag_agent.answer(query_simple_with_context)

conversation += [query_simple_as_message, answer_simple]

print("\nResponse:")
print(answer_simple.content[0].text)

### Domain Specific Question

In [ ]:
query_domain = "Which pallets in our portfolio have a third-party verified EPD?"
query_domain_as_message = LLMMessage(
    content=[MessageContent(type="text", text=query_domain)], role=Roles.USER
)
query_domain_with_context = QueryWithContext(query=query_domain, history=conversation)

answer_domain = await rag_agent.answer(query_domain_with_context)

conversation += [query_domain_as_message, answer_domain]

print("\nResponse:")
print(answer_domain.content[0].text)

---

## Test Memory

A follow-up query asks the agent to summarize the conversation. No retrieval tool is needed, the agent draws on the history passed in `conversation`.

In [ ]:
query = "Summarize the conversation (some words per message)?"
query_as_message = LLMMessage(
    content=[MessageContent(type="text", text=query)], role=Roles.USER
)
query_with_context = QueryWithContext(query=query, history=conversation)

answer = await rag_agent.answer(query_with_context)

print("\nResponse:")
print(answer.content[0].text)

----------------